In [3]:
!pip install -q -U osmnx geopandas folium networkx shapely google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 783.6/783.6 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.6/240.6 kB 18.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.49.2 which is incompatible.


In [4]:

import osmnx as ox
import networkx as nx
import folium
from IPython.display import display
from google import genai
import warnings
warnings.filterwarnings('ignore')

print("ADIM 1: OpenStreetMap Uzamsal Veri Çekimi")
place_name = "Beyoğlu, Istanbul, Turkey"

# Yaya yol ağını graf (network) olarak çekiyoruz
G = ox.graph_from_place(place_name, network_type='walk')

print(f"✅ Başarılı! Beyoğlu yaya ağı topolojisi indirildi.")
print(f"Sisteme Yüklenen Düğüm (Kavşak) Sayısı: {len(G.nodes)}")

ADIM 1: OpenStreetMap Uzamsal Veri Çekimi
✅ Başarılı! Beyoğlu yaya ağı topolojisi indirildi.
Sisteme Yüklenen Düğüm (Kavşak) Sayısı: 4567


In [5]:
import getpass
from google import genai

print("ADIM 2: GeoAI Ajanı Kurulumu")
print("Sistemi başlatmak için lütfen kendi Google Gemini API anahtarınızı girin:")

# Kullanıcıdan API anahtarını güvenli bir şekilde (ekranda göstermeden) istiyoruz
GOOGLE_API_KEY = getpass.getpass("API Key (Gizli): ")
client = genai.Client(api_key=GOOGLE_API_KEY)

print("✅ Başarılı! Gemini API bağlantısı kuruldu ve otonom ajan göreve hazır.")

ADIM 2: GeoAI Ajanı Kurulumu
Sistemi başlatmak için lütfen kendi Google Gemini API anahtarınızı girin:
API Key (Gizli): ··········
✅ Başarılı! Gemini API bağlantısı kuruldu ve otonom ajan göreve hazır.


In [11]:
import geopandas as gpd
from shapely.geometry import Point

print("--- AŞAMA 1: GÖKYÜZÜNE DOKUNANLAR ---")

# Koordinatlar: Taksim Meydanı -> Galata Kulesi
start_lat, start_lon = 41.0369, 28.9850
target_lat, target_lon = 41.0256, 28.9741
hedef_adi = "Galata Kulesi"

# 1. LLM ile Otonom Bilmece Üretimi
prompt = f"Sen mistik bir GeoAI ajansın. Oyuncunun hedefi {hedef_adi}. Doğrudan adını vermeden, hedefin tarihi ve fiziksel özelliklerini içeren 2 cümlelik mistik bir bilmece yaz."

try:
    response = client.models.generate_content(model='gemini-2.0-flash', contents=prompt)
    bilmece = response.text
except Exception as e:
    bilmece = "Kollarına rüzgarı, kalbine cesareti takan o tarihi adamın gökyüzüne dokunduğu taş kuleyi bul. Şehre yukarıdan bak, rüzgarın fısıldadığı o büyük atılımı dinle."

print("📜 AJANIN BİLMECESİ:\n", bilmece)

# 2. Etkileşim
cevap = input("\nTahmin ettiğin hedef nedir? ").strip().lower()

if "galata" in cevap:
    print("\n✅ DOĞRU! Uzamsal ağ analizi ve Mekansal Tampon (Buffer) hesaplanıyor...")

    # 3. NetworkX ile Ağ Analizi (En Kısa Yol)
    orig = ox.distance.nearest_nodes(G, X=start_lon, Y=start_lat)
    dest = ox.distance.nearest_nodes(G, X=target_lon, Y=target_lat)
    route = ox.shortest_path(G, orig, dest, weight='length')

    # 4. Mekansal Tampon (Buffer) Analizi Şovu!
    # Mühendislik detayı: Doğru metre hesabı için derece (WGS84) -> UTM projeksiyon dönüşümü yapıyoruz
    hedef_nokta = gpd.GeoSeries([Point(target_lon, target_lat)], crs="EPSG:4326")
    hedef_utm = hedef_nokta.to_crs(hedef_nokta.estimate_utm_crs())

    # Hedefin etrafına 300 metrelik buffer atıyoruz ve harita için tekrar WGS84'e çeviriyoruz
    buffer_300m_wgs84 = hedef_utm.buffer(300).to_crs("EPSG:4326")

    # 5. Folium ile Görselleştirme
    m1 = folium.Map(location=[(start_lat+target_lat)/2, (start_lon+target_lon)/2], zoom_start=15)

    # Rotayı çiz
    route_coords = [(G.nodes[node]['y'], G.nodes[node]['x']) for node in route]
    folium.PolyLine(locations=route_coords, color="blue", weight=5, tooltip="Otonom Rota").add_to(m1)

    # 300m Buffer Alanını Haritaya Ekle (Turuncu renkli)
    folium.GeoJson(
        buffer_300m_wgs84,
        name="300m Etki Alanı",
        style_function=lambda x: {'color': 'orange', 'fillOpacity': 0.3}
    ).add_to(m1)

    # Başlangıç ve Hedef işaretçilerini ekle
    folium.Marker([start_lat, start_lon], popup="Başlangıç", icon=folium.Icon(color="green")).add_to(m1)
    folium.Marker([target_lat, target_lon], popup=hedef_adi, icon=folium.Icon(color="red", icon="star")).add_to(m1)

    print("🎁 ÖDÜL: Kuleyi en iyi açıdan çekebileceğin gizli terasın kilidini açtın!\n")

    display(m1)
    m1.save("asama1_harita.html")
else:
    print("❌ Yanlış cevap. Lütfen bu kod hücresini tekrar çalıştırıp yeniden tahmin et.")

--- AŞAMA 1: GÖKYÜZÜNE DOKUNANLAR ---
📜 AJANIN BİLMECESİ:
 Kollarına rüzgarı, kalbine cesareti takan o tarihi adamın gökyüzüne dokunduğu taş kuleyi bul. Şehre yukarıdan bak, rüzgarın fısıldadığı o büyük atılımı dinle.

Tahmin ettiğin hedef nedir? galata kulesi

✅ DOĞRU! Uzamsal ağ analizi ve Mekansal Tampon (Buffer) hesaplanıyor...
🎁 ÖDÜL: Kuleyi en iyi açıdan çekebileceğin gizli terasın kilidini açtın!



In [13]:
import geopandas as gpd
from shapely.geometry import Point

print("--- AŞAMA 2: SANATIN İZİNDE ---")

# Koordinatlar: Galata Kulesi -> Pera Müzesi
start_lat, start_lon = 41.0256, 28.9741
target_lat, target_lon = 41.0318, 28.9748
hedef_adi = "Pera Müzesi"

# 1. LLM ile Otonom Bilmece Üretimi (Hata Tolere Etme / B Planı Entegreli)
prompt = f"Sen mistik bir GeoAI ajansın. Oyuncunun hedefi {hedef_adi}. Adını vermeden, içinde 'Kaplumbağa Terbiyecisi' gibi ünlü eserlerin sergilendiği bu tarihi sanat merkezini anlatan 2 cümlelik mistik bir bilmece yaz."

try:
    response = client.models.generate_content(model='gemini-2.0-flash', contents=prompt)
    bilmece = response.text
except Exception as e:
    bilmece = "Kaplumbağaları sabırla terbiye eden o meşhur adamın fırça darbelerini saklayan tarihi binayı bul. Sanatın ve tarihin kesiştiği bu görkemli kapıdan içeri gir."

print("📜 AJANIN BİLMECESİ:\n", bilmece)

# 2. Etkileşim
cevap = input("\nTahmin ettiğin hedef nedir? ").strip().lower()

if "pera" in cevap or "müze" in cevap:
    print("\n✅ DOĞRU! Uzamsal ağ analizi ve Mekansal Tampon (Buffer) hesaplanıyor...")

    # 3. NetworkX ile Ağ Analizi
    orig = ox.distance.nearest_nodes(G, X=start_lon, Y=start_lat)
    dest = ox.distance.nearest_nodes(G, X=target_lon, Y=target_lat)
    route = ox.shortest_path(G, orig, dest, weight='length')

    # 4. Mekansal Tampon (Buffer) Analizi
    # WGS84 -> UTM Dönüşümü ve 300m Tampon Hesaplama
    hedef_nokta = gpd.GeoSeries([Point(target_lon, target_lat)], crs="EPSG:4326")
    hedef_utm = hedef_nokta.to_crs(hedef_nokta.estimate_utm_crs())
    buffer_300m_wgs84 = hedef_utm.buffer(300).to_crs("EPSG:4326")

    # 5. Folium ile Görselleştirme
    m2 = folium.Map(location=[(start_lat+target_lat)/2, (start_lon+target_lon)/2], zoom_start=16)

    # Rotayı çiz
    route_coords = [(G.nodes[node]['y'], G.nodes[node]['x']) for node in route]
    folium.PolyLine(locations=route_coords, color="purple", weight=5, tooltip="2. Aşama Rotası").add_to(m2)

    # 300m Buffer Alanını Ekle (Turuncu)
    folium.GeoJson(
        buffer_300m_wgs84,
        name="300m Etki Alanı",
        style_function=lambda x: {'color': 'orange', 'fillOpacity': 0.2, 'weight': 2}
    ).add_to(m2)

    # İşaretçileri ekle
    folium.Marker([start_lat, start_lon], popup="Galata Kulesi (Eski Konum)", icon=folium.Icon(color="gray")).add_to(m2)
    folium.Marker([target_lat, target_lon], popup=hedef_adi, icon=folium.Icon(color="red", icon="star")).add_to(m2)

    print("🎁 ÖDÜL: Müzenin en üst katındaki eşsiz Haliç manzaralı gizli kafenin konumunu kazandın!\n")

    display(m2)
    m2.save("asama2_harita.html")
else:
    print("❌ Yanlış cevap. Lütfen bu kod hücresini tekrar çalıştırıp yeniden tahmin et.")

--- AŞAMA 2: SANATIN İZİNDE ---
📜 AJANIN BİLMECESİ:
 Kaplumbağaları sabırla terbiye eden o meşhur adamın fırça darbelerini saklayan tarihi binayı bul. Sanatın ve tarihin kesiştiği bu görkemli kapıdan içeri gir.

Tahmin ettiğin hedef nedir? pera müzesi

✅ DOĞRU! Uzamsal ağ analizi ve Mekansal Tampon (Buffer) hesaplanıyor...
🎁 ÖDÜL: Müzenin en üst katındaki eşsiz Haliç manzaralı gizli kafenin konumunu kazandın!



In [16]:
import geopandas as gpd
from shapely.geometry import Point

print("--- AŞAMA 3: KÜLLERİNDEN DOĞAN GEÇİT ---")

# Koordinatlar: Pera Müzesi -> Çiçek Pasajı
start_lat, start_lon = 41.0318, 28.9748
target_lat, target_lon = 41.0343, 28.9774
hedef_adi = "Çiçek Pasajı"

# 1. LLM ile Otonom Bilmece Üretimi (Hata Tolere Etme / B Planı Entegreli)
prompt = f"Sen mistik bir GeoAI ajansın. Oyuncunun hedefi {hedef_adi}. Adını vermeden, İstiklal Caddesi'ndeki bu tarihi, cam kubbeli ve bir zamanlar tiyatroya ev sahipliği yapmış olan görkemli geçidi anlatan 2 cümlelik destansı bir bilmece yaz."

try:
    response = client.models.generate_content(model='gemini-2.0-flash', contents=prompt)
    bilmece = response.text
except Exception as e:
    bilmece = "Bir zamanlar sahnelerinde dramların oynandığı, küllerinden doğup cam kubbeleriyle göğü kucaklayan o süslü geçidi bul. Şehrin kalbinde, neşenin ve tarihin yankılandığı o görkemli avluya gir."

print("📜 AJANIN BİLMECESİ:\n", bilmece)

# 2. Etkileşim
cevap = input("\nTahmin ettiğin hedef nedir? ").strip().lower()

if "çiçek" in cevap or "pasaj" in cevap:
    print("\n✅ DOĞRU! Uzamsal ağ analizi ve Mekansal Tampon (Buffer) hesaplanıyor...")

    # 3. NetworkX ile Ağ Analizi
    orig = ox.distance.nearest_nodes(G, X=start_lon, Y=start_lat)
    dest = ox.distance.nearest_nodes(G, X=target_lon, Y=target_lat)
    route = ox.shortest_path(G, orig, dest, weight='length')

    # 4. Mekansal Tampon (Buffer) Analizi
    hedef_nokta = gpd.GeoSeries([Point(target_lon, target_lat)], crs="EPSG:4326")
    hedef_utm = hedef_nokta.to_crs(hedef_nokta.estimate_utm_crs())
    buffer_200m_wgs84 = hedef_utm.buffer(200).to_crs("EPSG:4326")

    # 5. Folium ile Görselleştirme
    m3 = folium.Map(location=[(start_lat+target_lat)/2, (start_lon+target_lon)/2], zoom_start=16)

    # Rotayı çiz
    route_coords = [(G.nodes[node]['y'], G.nodes[node]['x']) for node in route]
    folium.PolyLine(locations=route_coords, color="gold", weight=5, tooltip="3. Aşama Rotası").add_to(m3)

    # 200m Buffer Alanını Ekle (Altın Sarısı)
    folium.GeoJson(
        buffer_200m_wgs84,
        name="200m Etki Alanı",
        style_function=lambda x: {'color': 'gold', 'fillOpacity': 0.2, 'weight': 2}
    ).add_to(m3)

    # İşaretçileri ekle
    folium.Marker([start_lat, start_lon], popup="Pera Müzesi (Eski Konum)", icon=folium.Icon(color="gray")).add_to(m3)
    folium.Marker([target_lat, target_lon], popup=hedef_adi, icon=folium.Icon(color="orange", icon="star")).add_to(m3)

    print("🎁 ÖDÜL: Pasajın tarihindeki ilk tiyatro oyununun dijital biletini arşivinden çıkardın!\n")

    display(m3)
    m3.save("asama3_harita.html")
else:
    print("❌ Yanlış cevap. Lütfen hücreyi tekrar çalıştır.")

--- AŞAMA 3: KÜLLERİNDEN DOĞAN GEÇİT ---
📜 AJANIN BİLMECESİ:
 Bir zamanlar sahnelerinde dramların oynandığı, küllerinden doğup cam kubbeleriyle göğü kucaklayan o süslü geçidi bul. Şehrin kalbinde, neşenin ve tarihin yankılandığı o görkemli avluya gir.

Tahmin ettiğin hedef nedir? çiçek pasajı

✅ DOĞRU! Uzamsal ağ analizi ve Mekansal Tampon (Buffer) hesaplanıyor...
🎁 ÖDÜL: Pasajın tarihindeki ilk tiyatro oyununun dijital biletini arşivinden çıkardın!



In [17]:
import geopandas as gpd
from shapely.geometry import Point

print("--- AŞAMA 4: USTANIN ESERİ (FİNAL) ---")

# Koordinatlar: Çiçek Pasajı -> Kılıç Ali Paşa Camii (Tophane)
start_lat, start_lon = 41.0343, 28.9774
target_lat, target_lon = 41.0267, 28.9811
hedef_adi = "Kılıç Ali Paşa Camii"

# 1. LLM ile Otonom Bilmece Üretimi (Hata Tolere Etme / B Planı Entegreli)
prompt = f"Sen mistik bir GeoAI ajansın. Oyuncunun final hedefi {hedef_adi}. Adını vermeden, Tophane sahilinde yer alan, Mimar Sinan'ın kusursuz mühendislik dehasını taşıyan bu tarihi yapıyı anlatan 2 cümlelik destansı bir bilmece yaz."

try:
    response = client.models.generate_content(model='gemini-2.0-flash', contents=prompt)
    bilmece = response.text
except Exception as e:
    bilmece = "Denizin kokusunu taşıyan, büyük ustanın kubbelere mühendislik fısıldadığı o asırlık mabedi bul. Yokuşları in ve martı seslerinin karıştığı o kusursuz avluya adım at."

print("📜 AJANIN BİLMECESİ:\n", bilmece)

# 2. Etkileşim
cevap = input("\nTahmin ettiğin hedef nedir? ").strip().lower()

if "kılıç" in cevap or "ali" in cevap or "cami" in cevap:
    print("\n✅ DOĞRU! Final rotası ve Mekansal Tampon (Buffer) hesaplanıyor...")

    # 3. NetworkX ile Ağ Analizi
    orig = ox.distance.nearest_nodes(G, X=start_lon, Y=start_lat)
    dest = ox.distance.nearest_nodes(G, X=target_lon, Y=target_lat)
    route = ox.shortest_path(G, orig, dest, weight='length')

    # 4. Mekansal Tampon (Buffer) Analizi
    hedef_nokta = gpd.GeoSeries([Point(target_lon, target_lat)], crs="EPSG:4326")
    hedef_utm = hedef_nokta.to_crs(hedef_nokta.estimate_utm_crs())
    buffer_300m_wgs84 = hedef_utm.buffer(300).to_crs("EPSG:4326")

    # 5. Folium ile Görselleştirme
    m4 = folium.Map(location=[(start_lat+target_lat)/2, (start_lon+target_lon)/2], zoom_start=15)

    # Rotayı çiz
    route_coords = [(G.nodes[node]['y'], G.nodes[node]['x']) for node in route]
    folium.PolyLine(locations=route_coords, color="darkred", weight=6, tooltip="Final Rotası").add_to(m4)

    # 300m Buffer Alanını Ekle (Camgöbeği/Turkuaz)
    folium.GeoJson(
        buffer_300m_wgs84,
        name="300m Etki Alanı",
        style_function=lambda x: {'color': 'cyan', 'fillOpacity': 0.2, 'weight': 2}
    ).add_to(m4)

    # İşaretçileri ekle
    folium.Marker([start_lat, start_lon], popup="Çiçek Pasajı (Eski Konum)", icon=folium.Icon(color="gray")).add_to(m4)
    folium.Marker([target_lat, target_lon], popup=hedef_adi, icon=folium.Icon(color="darkred", icon="star")).add_to(m4)

    print("🏆 BÜYÜK ÖDÜL: Mekansal analiz sınırlarını aştın! 'Usta Mekan Avcısı' rozetini kazandın!\n")

    display(m4)
    m4.save("asama4_harita.html")
else:
    print("❌ Yanlış cevap. Lütfen hücreyi tekrar çalıştır.")

--- AŞAMA 4: USTANIN ESERİ (FİNAL) ---
📜 AJANIN BİLMECESİ:
 Denizin kokusunu taşıyan, büyük ustanın kubbelere mühendislik fısıldadığı o asırlık mabedi bul. Yokuşları in ve martı seslerinin karıştığı o kusursuz avluya adım at.

Tahmin ettiğin hedef nedir? Kılıç ali paşa camii

✅ DOĞRU! Final rotası ve Mekansal Tampon (Buffer) hesaplanıyor...
🏆 BÜYÜK ÖDÜL: Mekansal analiz sınırlarını aştın! 'Usta Mekan Avcısı' rozetini kazandın!

